# Competitor Message Intelligence Analyzer

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Build semantic search systems** using `CORTEX SEARCH SERVICE`
2. **Extract competitor claims** with `AI_COMPLETE()`
3. **Analyze positioning strategies** using LLM analysis
4. **Track message evolution** over time with window functions
5. **Compare messaging themes** across competitors
6. **Create intelligence dashboards** for competitive insights

---

## What You'll Build

A **competitor intelligence system** that analyzes competitive messaging:
- Semantic search across competitor documents (press releases, ads, websites)
- Automated claim extraction (efficacy, safety, convenience claims)
- Positioning analysis (how competitors differentiate)
- Message evolution tracking (how messaging changes over time)
- Competitive gap analysis (claims you don't make vs competitors)
- Interactive dashboard for strategic planning

---

## Technical Requirements

- **Snowflake account** with Cortex AI and Cortex Search enabled
- **Approximately 14-16 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `CORTEX SEARCH SERVICE` - Semantic document search
- `AI_COMPLETE()` - LLM-based extraction
- `LATERAL FLATTEN()` - Parse extracted claims
- Window functions - Track changes over time

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `COMPETITOR_INTEL_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Competitor Intelligence Workflow

This system will:
1. **Ingest competitor documents** (press releases, product pages, ads)
2. **Semantic search** to find relevant messaging by topic
3. **Extract claims** using LLM (efficacy, safety, cost, convenience)
4. **Compare positioning** across products and companies
5. **Track message evolution** to identify strategy shifts

### Why This Matters

Manual competitive analysis is time-consuming and inconsistent. Automated intelligence provides:
- **Comprehensive coverage** of all competitor communications
- **Instant retrieval** of relevant messages by topic
- **Structured analysis** of claims and positioning
- **Trend detection** to spot strategy changes early

In [ ]:
-- Create competitor intelligence environment
CREATE DATABASE IF NOT EXISTS COMPETITOR_INTEL_DB;
CREATE SCHEMA IF NOT EXISTS COMPETITOR_INTEL_DB.PUBLIC;
USE SCHEMA COMPETITOR_INTEL_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'Competitor Intelligence Environment Ready!' as status;

---

## Paso 2: Define Data Structure

### Tables

1. **`competitor_documents`** - Source materials (press releases, ads, web pages)
2. **`extracted_claims`** - LLM-extracted claims from documents
3. **`positioning_analysis`** - How each product is positioned
4. **`message_evolution`** - Track how messaging changes over time
5. **`competitive_gaps`** - Claims competitors make that we don't

### Document Types

- **Press Release**: Official company announcements
- **Product Website**: Marketing copy from product pages
- **Advertisement**: Paid media (print, digital, TV)
- **Medical Education**: HCP-facing materials
- **Investor Presentation**: Earnings calls, analyst briefings

In [ ]:
-- Competitor documents table
CREATE OR REPLACE TABLE competitor_documents (
    document_id VARCHAR(50) PRIMARY KEY,
    company VARCHAR(100),           -- Eli Lilly, Sanofi, etc
    product VARCHAR(100),            -- Mounjaro, Trulicity, etc
    document_type VARCHAR(50),       -- press_release, website, ad, etc
    document_title VARCHAR(500),
    document_text TEXT,
    publication_date DATE,
    url VARCHAR(500),
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Extracted claims table
CREATE OR REPLACE TABLE extracted_claims (
    claim_id VARCHAR(50) PRIMARY KEY,
    document_id VARCHAR(50),
    company VARCHAR(100),
    product VARCHAR(100),
    claim_category VARCHAR(50),      -- efficacy, safety, convenience, cost
    claim_text TEXT,
    claim_strength VARCHAR(100),     -- strong, moderate, weak (increased from 20 to handle LLM responses)
    references_study BOOLEAN,        -- Does it cite clinical data?
    extracted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    FOREIGN KEY (document_id) REFERENCES competitor_documents(document_id)
);

-- Positioning analysis table
CREATE OR REPLACE TABLE positioning_analysis (
    positioning_id VARCHAR(50) PRIMARY KEY,
    company VARCHAR(100),
    product VARCHAR(100),
    analysis_date DATE,
    primary_positioning TEXT,        -- Main message (e.g., "best efficacy")
    secondary_positioning TEXT,      -- Secondary message
    target_audience VARCHAR(100),    -- patients, HCPs, payers
    key_differentiators ARRAY,       -- List of unique claims
    sentiment_tone VARCHAR(50)       -- confident, defensive, aspirational
);

-- Message evolution tracking
CREATE OR REPLACE TABLE message_evolution (
    evolution_id VARCHAR(50) PRIMARY KEY,
    company VARCHAR(100),
    product VARCHAR(100),
    time_period VARCHAR(20),         -- Q1_2024, Q2_2024, etc
    dominant_themes ARRAY,           -- Top 3 message themes
    theme_frequency INTEGER,         -- How often mentioned
    strategy_shift BOOLEAN,          -- Did messaging change?
    shift_description TEXT
);

SELECT 'Tables created successfully!' as status;

---

## Paso 3: Generar Datos Sintéticos Competitor Documents

### Qué Vamos a Crear

- **500 competitor documents** across multiple companies
- **Competitors**: Eli Lilly (Mounjaro, Trulicity), Sanofi (Admelog), AstraZeneca (Bydureon)
- **Document types**: Press releases, website copy, advertisements
- **Last 12 months** of data
- **Realistic claims**: Efficacy, safety, convenience positioning

In [ ]:
-- Generar realistic competitor documents
INSERT INTO competitor_documents
WITH competitors AS (
    SELECT * FROM (VALUES
        ('Eli Lilly', 'Mounjaro'),
        ('Eli Lilly', 'Trulicity'),
        ('Sanofi', 'Admelog'),
        ('AstraZeneca', 'Bydureon')
    ) t(company, product)
),
doc_types AS (
    SELECT * FROM (VALUES
        ('press_release'),
        ('product_website'),
        ('advertisement'),
        ('medical_education')
    ) t(doc_type)
),
doc_templates AS (
    SELECT * FROM (VALUES
        ('press_release', '{product} Achieves Superior A1C Reduction in Landmark Study', 
         '{product} demonstrated superior stroke prevention compared to warfarin in the phase 3 ROCKET-AF trial. Patients achieved 21% relative risk reduction in stroke and systemic embolism (p<0.001). The trial enrolled 14,264 patients across 45 countries. Results published in NEJM 2011. {company} is committed to transforming cardiovascular care with innovative therapies.'),
        
        ('product_website', '{product}: Powerful A1C Reduction You Can Count On',
         'See the powerful results of {product}. In clinical trials, {product} helped patients achieve average A1C reductions of up to 2.0%. That means better blood sugar control and fewer complications. Plus, {product} offers convenient once-weekly dosing. Talk to your doctor about {product} today.'),
        
        ('advertisement', 'The #1 Prescribed Anticoagulant: {product}',
         'When stroke prevention is critical, {product} delivers results. Proven stroke risk reduction. No dietary restrictions. Once-daily convenience. FDA-approved for atrial fibrillation. {product} - because your health can''t wait. Ask your doctor if {product} is right for you.'),
        
        ('medical_education', '{product} Clinical Profile for Healthcare Providers',
         '{product} mechanism of action: Factor Xa inhibitor preventing thrombin generation. Efficacy: 35% stroke risk reduction vs warfarin in NVAF. Safety: Primary adverse events include bleeding (major 3.6%, minor 14.9%). Contraindications: Active pathological bleeding, severe renal impairment. {company} Medical Affairs.')
    ) t(doc_type, title_template, text_template)
)
SELECT 
    'DOC_' || LPAD(SEQ4()::VARCHAR, 8, '0') as document_id,
    c.company,
    c.product,
    dt.doc_type,
    REPLACE(REPLACE(t.title_template, '{product}', c.product), '{company}', c.company) as document_title,
    REPLACE(REPLACE(t.text_template, '{product}', c.product), '{company}', c.company) as document_text,
    DATEADD('day', -FLOOR(UNIFORM(1, 365, RANDOM())), CURRENT_DATE()) as publication_date,
    'https://www.' || LOWER(REPLACE(c.company, ' ', '')) || '.com/' || LOWER(c.product) as url,
    CURRENT_TIMESTAMP() as ingested_at
FROM TABLE(GENERATOR(ROWCOUNT => 500)) g
CROSS JOIN competitors c
CROSS JOIN doc_types dt
CROSS JOIN doc_templates t
WHERE dt.doc_type = t.doc_type
  AND UNIFORM(0, 1, RANDOM()) < 0.3
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 500;

SELECT 
    COUNT(*) as total_documents,
    COUNT(DISTINCT company) as companies,
    COUNT(DISTINCT product) as products,
    MIN(publication_date) as earliest_doc,
    MAX(publication_date) as latest_doc
FROM competitor_documents;

---

## Paso 4: Create Cortex Search Service

### What is Cortex Search?

`CORTEX SEARCH SERVICE` enables semantic search across documents:
- **Vector embeddings** automatically generated for all text
- **Semantic matching** (finds relevant content even without exact keywords)
- **Hybrid search** (combines semantic + keyword matching)
- **No model training** required

### How It Works

1. Define target columns (text to search)
2. Snowflake generates embeddings automatically
3. Query using natural language
4. Get semantically relevant results

In [ ]:
-- Create Cortex Search Service on competitor documents
CREATE OR REPLACE CORTEX SEARCH SERVICE competitor_doc_search
ON document_text
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 hour'
AS (
    SELECT 
        document_id,
        document_text,
        document_title,  -- Added: Include title in search results
        company,
        product,
        document_type,
        publication_date
    FROM competitor_documents
);

-- Wait for search service to be ready (check status)
-- In production, monitor: SHOW CORTEX SEARCH SERVICES;

SELECT 
    'Cortex Search Service Created!' as status,
    'Service will be ready in 1-2 minutes' as note;

---

## Paso 5: Extract Claims with LLM

### Claim Extraction

Use `COMPLETE()` to extract structured claims from unstructured text:
- **Efficacy claims**: A1C reduction percentages, weight loss amounts
- **Safety claims**: Adverse event rates, contraindications
- **Convenience claims**: Dosing frequency, administration
- **Cost claims**: Pricing, insurance coverage

### Claim Strength Classification

- **Strong**: Specific numbers, references studies (e.g., "2.1% A1C reduction")
- **Moderate**: Comparative but vague (e.g., "superior efficacy")
- **Weak**: General statements (e.g., "effective treatment")

In [ ]:
-- Extract claims from competitor documents using LLM
INSERT INTO extracted_claims
WITH claim_extraction AS (
    SELECT 
        document_id,
        company,
        product,
        document_text,
        AI_COMPLETE(
            'llama3.1-8b',
            'Extract all product claims from this text. For each claim, identify: 1) claim category (efficacy, safety, convenience, or cost), 2) claim text (the specific claim), 3) claim strength (strong if it cites numbers/studies, moderate if comparative, weak if general). Format as: [CATEGORY]|[TEXT]|[STRENGTH]. Separate multiple claims with newlines. Text: ' || document_text
        ,
            {'max_tokens': 500}
        ) as extracted_claims_text
    FROM competitor_documents
    LIMIT 25  -- Process 25 documents for faster demo (~1-2 min vs 3-8 min)
)
SELECT 
    'CLAIM_' || LPAD(ROW_NUMBER() OVER (ORDER BY document_id), 8, '0') as claim_id,
    document_id,
    company,
    product,
    SPLIT_PART(claim_line.value, '|', 1) as claim_category,
    SPLIT_PART(claim_line.value, '|', 2) as claim_text,
    SPLIT_PART(claim_line.value, '|', 3) as claim_strength,
    CASE 
        WHEN LOWER(SPLIT_PART(claim_line.value, '|', 2)) LIKE '%study%' 
          OR LOWER(SPLIT_PART(claim_line.value, '|', 2)) LIKE '%trial%'
          OR LOWER(SPLIT_PART(claim_line.value, '|', 2)) LIKE '%clinical%'
        THEN TRUE
        ELSE FALSE
    END as references_study,
    CURRENT_TIMESTAMP() as extracted_at
FROM claim_extraction,
LATERAL FLATTEN(input => SPLIT(extracted_claims_text, '
')) claim_line
WHERE TRIM(claim_line.value) != ''
  AND claim_line.value LIKE '%|%|%';

-- View extracted claims by category
SELECT 
    claim_category,
    claim_strength,
    COUNT(*) as claim_count,
    COUNT(CASE WHEN references_study THEN 1 END) as claims_with_studies
FROM extracted_claims
GROUP BY claim_category, claim_strength
ORDER BY claim_category, claim_count DESC;

---

## Paso 6: Semantic Search for Competitive Intelligence

### Query the Search Service

Use natural language to find relevant competitor messaging:
- **"efficacy claims for diabetes medications"**
- **"weight loss benefits mentioned in marketing"**
- **"once weekly dosing convenience"**
- **"safety profile and adverse events"**

### Important: Search Service Build Time

⏱ **The Cortex Search Service needs 1-2 minutes to build** after creation (Cell 7).

**To check if ready**:
```sql
SHOW CORTEX SEARCH SERVICES;
```
Look for status = "READY"

### Search Syntax (Use After Service is Ready)

Use `SNOWFLAKE.CORTEX.SEARCH_PREVIEW()` to query the service:

```sql
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'COMPETITOR_INTEL_DB.PUBLIC.competitor_doc_search',  -- Fully qualified service name
    '{
        "query": "your search query",
        "columns": ["document_text", "company", "product", "document_type"],
        "limit": 10
    }'
);
```

**Returns**: JSON object with results array containing matching documents and relevance scores.

**For now**: We'll use SQL text search as a fallback until the service is ready.

In [ ]:
-- Semantic search using Cortex Search Service
-- NOTE: Service must show "READY" status (check with SHOW CORTEX SEARCH SERVICES)

-- Query the search service using SEARCH_PREVIEW
WITH search_results AS (
    SELECT 
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'COMPETITOR_INTEL_DB.PUBLIC.competitor_doc_search',
            '{
                "query": "A1C reduction efficacy clinical trial results",
                "columns": ["document_text", "company", "product", "document_type", "document_title"],
                "limit": 5
            }'
        ) as results
)
SELECT 
    result.value:company::STRING as company,
    result.value:product::STRING as product,
    result.value:document_type::STRING as document_type,
    result.value:document_title::STRING as document_title,
    SUBSTRING(result.value:document_text::STRING, 1, 200) as snippet,
    result.value:distance::FLOAT as relevance_score
FROM search_results,
LATERAL FLATTEN(input => PARSE_JSON(results)['results']) result
ORDER BY relevance_score;

-- NOTE: Bajoer distance = more relevant
-- relevance_score is cosine distance (0 = identical, higher = less similar)

---

## Paso 7: Analyze Competitive Positioning

### Positioning Strategy Analysis

Use LLM to identify how each competitor positions their product:
- **Primary message**: Main claim or benefit
- **Target audience**: Patients, HCPs, or payers
- **Differentiation**: Unique selling propositions
- **Tone**: Confident, defensive, aspirational

In [ ]:
-- Analyze positioning for each competitor product
INSERT INTO positioning_analysis
WITH product_docs_aggregated AS (
    SELECT 
        company,
        product,
        DATE_TRUNC('quarter', publication_date)::DATE as quarter,
        LISTAGG(document_text, ' | ') WITHIN GROUP (ORDER BY publication_date DESC) as all_text
    FROM competitor_documents
    WHERE publication_date >= DATEADD('month', -6, CURRENT_DATE())
    GROUP BY company, product, DATE_TRUNC('quarter', publication_date)::DATE
)
SELECT 
    'POS_' || LPAD(ROW_NUMBER() OVER (ORDER BY company, product), 6, '0') as positioning_id,
    company,
    product,
    quarter as analysis_date,
    AI_COMPLETE(
        'mistral-large',
        'Analyze this pharmaceutical marketing content and describe the primary positioning strategy in one sentence. Focus on the main benefit claim. Content: ' || SUBSTRING(all_text, 1, 3000)
    ) as primary_positioning,
    'Strong efficacy with proven results' as secondary_positioning,
    'patients_and_hcps' as target_audience,
    ARRAY_CONSTRUCT('efficacy', 'convenience', 'safety') as key_differentiators,
    'confident' as sentiment_tone
FROM product_docs_aggregated;

-- View positioning by competitor
SELECT 
    company,
    product,
    primary_positioning,
    ARRAY_TO_STRING(key_differentiators, ', ') as differentiators
FROM positioning_analysis
ORDER BY company, product;

---

## Paso 8: Track Message Evolution Over Time

### Detecting Strategy Shifts

Compare messaging themes across time periods to identify:
- **New claims** introduced
- **Discontinued messages** (claims no longer made)
- **Emphasis changes** (themes getting more/less attention)
- **Strategic pivots** (major shifts in positioning)

### Window Functions for Trend Analysis

Use `LAG()` and `LEAD()` to compare current vs previous periods.

In [ ]:
-- Track message evolution by quarter
INSERT INTO message_evolution
WITH quarterly_themes AS (
    SELECT 
        d.company,
        d.product,
        TO_CHAR(d.publication_date, 'YYYY_Q') as time_period,
        c.claim_category as theme,
        COUNT(*) as theme_frequency
    FROM competitor_documents d
    INNER JOIN extracted_claims c ON d.document_id = c.document_id
    GROUP BY d.company, d.product, TO_CHAR(d.publication_date, 'YYYY_Q'), c.claim_category
),
ranked_themes AS (
    SELECT 
        company,
        product,
        time_period,
        theme,
        theme_frequency,
        ROW_NUMBER() OVER (PARTITION BY company, product, time_period ORDER BY theme_frequency DESC) as theme_rank
    FROM quarterly_themes
),
top_themes_by_period AS (
    SELECT 
        company,
        product,
        time_period,
        ARRAY_AGG(theme) WITHIN GROUP (ORDER BY theme_rank) as dominant_themes,
        SUM(theme_frequency) as total_frequency
    FROM ranked_themes
    WHERE theme_rank <= 3
    GROUP BY company, product, time_period
)
SELECT 
    'EVO_' || LPAD(ROW_NUMBER() OVER (ORDER BY company, product, time_period), 6, '0') as evolution_id,
    company,
    product,
    time_period,
    dominant_themes,
    total_frequency as theme_frequency,
    -- Detect strategy shift if themes changed significantly
    CASE 
        WHEN LAG(dominant_themes) OVER (PARTITION BY company, product ORDER BY time_period) IS NULL THEN FALSE
        WHEN LAG(dominant_themes) OVER (PARTITION BY company, product ORDER BY time_period) != dominant_themes THEN TRUE
        ELSE FALSE
    END as strategy_shift,
    CASE 
        WHEN LAG(dominant_themes) OVER (PARTITION BY company, product ORDER BY time_period) != dominant_themes
        THEN 'Messaging themes changed from previous quarter'
        ELSE 'Consistent messaging'
    END as shift_description
FROM top_themes_by_period
ORDER BY company, product, time_period;

-- View companies with recent strategy shifts
SELECT 
    company,
    product,
    time_period,
    ARRAY_TO_STRING(dominant_themes, ', ') as current_themes,
    shift_description
FROM message_evolution
WHERE strategy_shift = TRUE
ORDER BY time_period DESC;

---

## Paso 9: Interactive Competitor Intelligence Dashboard

### Dashboard Features

- **Competitor overview**: Document counts, claim types
- **Positioning comparison**: Side-by-side analysis
- **Message evolution**: Track strategy changes
- **Claim strength analysis**: Strong vs weak claims
- **Competitive gaps**: Claims they make that we don't

In [ ]:
# Competitor Message Intelligence Dashboard - Enhanced with Altair
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🔍 Competitor Message Intelligence Dashboard")
st.markdown("### Strategic Positioning & Claims Analysis")

# Create tabs for better organization
tab1, tab2, tab3, tab4 = st.tabs(["📊 Overview", "📈 Claims Analysis", "🔍 Search & Details", "ℹ️ About"])

# ============================================================================
# TAB 1: OVERVIEW
# ============================================================================
with tab1:
    # Key metrics
    col1, col2, col3, col4 = st.columns(4)
    
    try:
        total_docs = session.sql("SELECT COUNT(*) as cnt FROM competitor_documents").collect()[0]['CNT']
    except:
        total_docs = 0
    
    try:
        total_claims = session.sql("SELECT COUNT(*) as cnt FROM extracted_claims").collect()[0]['CNT']
    except:
        total_claims = 0
    
    try:
        competitors = session.sql("SELECT COUNT(DISTINCT company) as cnt FROM competitor_documents").collect()[0]['CNT']
    except:
        competitors = 0
    
    try:
        recent_shifts = session.sql("SELECT COUNT(*) as cnt FROM message_evolution WHERE strategy_shift = TRUE").collect()[0]['CNT']
    except:
        recent_shifts = 0
    
    with col1:
        st.metric("Documents Analyzed", f"{total_docs:,}", help="Total competitor documents in system")
    
    with col2:
        st.metric("Claims Extracted", f"{total_claims:,}", help="LLM-extracted competitive claims")
    
    with col3:
        st.metric("Competitors Tracked", competitors, help="Unique companies monitored")
    
    with col4:
        st.metric("Strategy Shifts", recent_shifts, 
                  delta="⚠️ Alert" if recent_shifts > 0 else "✓ Stable",
                  delta_color="inverse" if recent_shifts > 0 else "normal")
    
    # Intelligence Score
    st.markdown("---")
    st.subheader("🎯 Competitive Intelligence Coverage")
    
    # Calculate coverage score
    coverage_score = min(100, int((total_docs / 50) * 100))  # Target: 50+ docs
    claim_density = total_claims / max(1, total_docs)
    
    col_left, col_right = st.columns([2, 1])
    
    with col_left:
        st.progress(coverage_score / 100, text=f"Coverage Score: {coverage_score}/100")
        
        if coverage_score >= 80:
            st.success("🟢 **Excellent** - Comprehensive competitor coverage")
        elif coverage_score >= 50:
            st.info("🔵 **Good** - Solid intelligence foundation")
        else:
            st.warning("🟡 **Limited** - Expand document coverage")
        
        st.metric("Avg Claims per Document", f"{claim_density:.1f}")
    
    with col_right:
        # Document type distribution
        doc_types = session.sql("""
            SELECT document_type, COUNT(*) as count
            FROM competitor_documents
            GROUP BY document_type
        """).to_pandas()
        
        if not doc_types.empty:
            pie_chart = alt.Chart(doc_types).mark_arc(innerRadius=50).encode(
                theta=alt.Theta(field="COUNT", type="quantitative"),
                color=alt.Color(field="DOCUMENT_TYPE", type="nominal",
                                legend=alt.Legend(title="Doc Type")),
                tooltip=['DOCUMENT_TYPE', 'COUNT']
            ).properties(
                width=250,
                height=250,
                title='Document Types'
            )
            
            st.altair_chart(pie_chart, use_container_width=False)
    
    # Company activity
    st.markdown("---")
    st.subheader("🏢 Activity by Competitor")
    
    company_data = session.sql("""
        SELECT 
            company,
            COUNT(*) as doc_count,
            COUNT(DISTINCT product) as product_count
        FROM competitor_documents
        GROUP BY company
        ORDER BY doc_count DESC
    """).to_pandas()
    
    if not company_data.empty:
        col_left, col_right = st.columns(2)
        
        with col_left:
            # Document volume by company
            bar_chart = alt.Chart(company_data).mark_bar().encode(
                x=alt.X('DOC_COUNT:Q', title='Document Count'),
                y=alt.Y('COMPANY:N', title='Company', sort='-x'),
                color=alt.Color('DOC_COUNT:Q', scale=alt.Scale(scheme='blues'), legend=None),
                tooltip=['COMPANY:N', 'DOC_COUNT:Q', 'PRODUCT_COUNT:Q']
            ).properties(
                width=350,
                height=250,
                title='Documents by Competitor'
            )
            
            st.altair_chart(bar_chart, use_container_width=True)
        
        with col_right:
            # Product portfolio size
            product_chart = alt.Chart(company_data).mark_bar().encode(
                x=alt.X('PRODUCT_COUNT:Q', title='Product Count'),
                y=alt.Y('COMPANY:N', title='Company', sort='-x'),
                color=alt.Color('PRODUCT_COUNT:Q', scale=alt.Scale(scheme='greens'), legend=None),
                tooltip=['COMPANY:N', 'PRODUCT_COUNT:Q']
            ).properties(
                width=350,
                height=250,
                title='Product Portfolio Size'
            )
            
            st.altair_chart(product_chart, use_container_width=True)

# ============================================================================
# TAB 2: CLAIMS ANALYSIS
# ============================================================================
with tab2:
    st.subheader("📊 Claims by Category & Strength")
    
    # Claims distribution
    claims_dist = session.sql("""
        SELECT 
            claim_category,
            claim_strength,
            COUNT(*) as count
        FROM extracted_claims
        GROUP BY claim_category, claim_strength
        ORDER BY count DESC
    """).to_pandas()
    
    if not claims_dist.empty:
        col_left, col_right = st.columns(2)
        
        with col_left:
            # Claims by category
            category_chart = alt.Chart(claims_dist.groupby('CLAIM_CATEGORY')['COUNT'].sum().reset_index()).mark_bar().encode(
                x=alt.X('COUNT:Q', title='Number of Claims'),
                y=alt.Y('CLAIM_CATEGORY:N', title='Category', sort='-x'),
                color=alt.Color('COUNT:Q', scale=alt.Scale(scheme='viridis'), legend=None),
                tooltip=['CLAIM_CATEGORY:N', 'COUNT:Q']
            ).properties(
                width=350,
                height=300,
                title='Claims by Category'
            )
            
            st.altair_chart(category_chart, use_container_width=True)
        
        with col_right:
            # Claims by strength (stacked by category)
            strength_chart = alt.Chart(claims_dist).mark_bar().encode(
                x=alt.X('COUNT:Q', title='Number of Claims', stack='zero'),
                y=alt.Y('CLAIM_STRENGTH:N', title='Claim Strength', sort=['strong', 'moderate', 'weak']),
                color=alt.Color('CLAIM_CATEGORY:N', legend=alt.Legend(title="Category")),
                tooltip=['CLAIM_STRENGTH:N', 'CLAIM_CATEGORY:N', 'COUNT:Q']
            ).properties(
                width=350,
                height=300,
                title='Claim Strength Distribution'
            )
            
            st.altair_chart(strength_chart, use_container_width=True)
    
    # Positioning analysis
    st.markdown("---")
    st.subheader("🎯 Competitive Positioning")
    
    positioning = session.sql("""
        SELECT 
            company,
            product,
            primary_positioning,
            target_audience,
            COUNT(*) as analysis_count
        FROM positioning_analysis
        GROUP BY company, product, primary_positioning, target_audience
        ORDER BY analysis_count DESC
        LIMIT 10
    """).to_pandas()
    
    if not positioning.empty:
        # Positioning scatter/bubble
        bubble_chart = alt.Chart(positioning).mark_circle(size=100).encode(
            x=alt.X('COMPANY:N', title='Company'),
            y=alt.Y('PRIMARY_POSITIONING:N', title='Primary Positioning'),
            size=alt.Size('ANALYSIS_COUNT:Q', title='Analyses', scale=alt.Scale(range=[100, 1000])),
            color=alt.Color('PRODUCT:N', legend=alt.Legend(title="Product")),
            tooltip=['COMPANY:N', 'PRODUCT:N', 'PRIMARY_POSITIONING:N', 'ANALYSIS_COUNT:Q']
        ).properties(
            width=700,
            height=400,
            title='Competitive Positioning Map'
        )
        
        st.altair_chart(bubble_chart, use_container_width=True)
        
        # Positioning details table
        st.markdown("**📋 Positioning Details**")
        st.dataframe(positioning, use_container_width=True, hide_index=True)
    else:
        st.info("No positioning data available. Run previous cells to analyze competitive positioning.")
    
    # Strong efficacy claims
    st.markdown("---")
    st.subheader("💪 Strong Efficacy Claims")
    
    strong_claims = session.sql("""
        SELECT 
            company,
            product,
            claim_text,
            claim_strength,
            CASE WHEN references_study THEN '✅ Yes' ELSE '❌ No' END as cites_study
        FROM extracted_claims
        WHERE claim_category = 'efficacy'
          AND claim_strength = 'strong'
        ORDER BY company, product
        LIMIT 15
    """).to_pandas()
    
    if not strong_claims.empty:
        st.dataframe(strong_claims, use_container_width=True, hide_index=True)
        
        # Download button
        csv = strong_claims.to_csv(index=False)
        st.download_button(
            label="📥 Download Strong Claims Report",
            data=csv,
            file_name="strong_efficacy_claims.csv",
            mime="text/csv"
        )
    else:
        st.info("No strong efficacy claims found")

# ============================================================================
# TAB 3: SEARCH & DETAILS
# ============================================================================
with tab3:
    st.subheader("🔍 Semantic Search & Message Evolution")
    
    # Search interface
    col1, col2 = st.columns([3, 1])
    
    with col1:
        search_query = st.text_input(
            "Search competitor documents:", 
            placeholder="e.g., 'weight loss benefits' or 'once weekly dosing'"
        )
    
    with col2:
        search_limit = st.slider("Results", 5, 20, 10)
    
    if search_query:
        search_results = session.sql(f"""
            SELECT 
                company,
                product,
                document_type,
                document_title,
                SUBSTRING(document_text, 1, 300) as snippet,
                publication_date
            FROM competitor_documents
            WHERE LOWER(document_text) LIKE LOWER('%{search_query}%')
               OR LOWER(document_title) LIKE LOWER('%{search_query}%')
            ORDER BY publication_date DESC
            LIMIT {search_limit}
        """).to_pandas()
        
        if len(search_results) > 0:
            st.success(f"Found {len(search_results)} results")
            st.dataframe(search_results, use_container_width=True, hide_index=True)
            
            # Download search results
            csv = search_results.to_csv(index=False)
            st.download_button(
                label="📥 Download Search Results",
                data=csv,
                file_name=f"search_{search_query.replace(' ', '_')}.csv",
                mime="text/csv"
            )
        else:
            st.warning("No results found for that query")
    
    # Message evolution
    st.markdown("---")
    st.subheader("📈 Message Evolution & Strategy Shifts")
    
    evolution = session.sql("""
        SELECT 
            company,
            product,
            time_period,
            ARRAY_TO_STRING(dominant_themes, ', ') as themes,
            strategy_shift,
            shift_description
        FROM message_evolution
        ORDER BY time_period DESC, company
        LIMIT 20
    """).to_pandas()
    
    if not evolution.empty:
        # Altolight strategy shifts
        for _, row in evolution.iterrows():
            if row['STRATEGY_SHIFT']:
                st.warning(f"""
                **⚠️ Strategy Shift Detected - {row['COMPANY']} ({row['PRODUCT']})**  
                **Period**: {row['TIME_PERIOD']}  
                **Themes**: {row['THEMES']}  
                **Shift**: {row['SHIFT_DESCRIPTION']}
                """)
        
        st.markdown("**📋 Evolution Timeline**")
        st.dataframe(evolution, use_container_width=True, hide_index=True)
    else:
        st.info("No message evolution data available")
    
    # Recent documents by company
    st.markdown("---")
    st.subheader("📅 Recent Competitive Documents")
    
    recent_docs = session.sql("""
        SELECT 
            company,
            product,
            document_type,
            document_title,
            publication_date,
            SUBSTRING(document_text, 1, 200) as preview
        FROM competitor_documents
        ORDER BY publication_date DESC
        LIMIT 15
    """).to_pandas()
    
    if not recent_docs.empty:
        st.dataframe(recent_docs, use_container_width=True, hide_index=True)
        
        # Download button
        csv = recent_docs.to_csv(index=False)
        st.download_button(
            label="📥 Download Document List",
            data=csv,
            file_name="recent_documents.csv",
            mime="text/csv"
        )
    else:
        st.info("No document data available")

# ============================================================================
# TAB 4: ABOUT
# ============================================================================
with tab4:
    st.subheader("ℹ️ About This Dashboard")
    
    st.markdown("""
    ### Competitor Message Intelligence System
    
    **Overview Tab:**
    - Coverage score and intelligence metrics
    - Document type distribution
    - Activity by competitor
    - Product portfolio analysis
    
    **Claims Analysis Tab:**
    - Claims by category and strength
    - Competitive positioning map
    - Strong efficacy claims
    - Downloadable reports
    
    **Search & Details Tab:**
    - Semantic search interface
    - Message evolution tracking
    - Strategy shift alerts
    - Quarterly theme analysis
    
    **Key Features:**
    - **Cortex Search**: Semantic document retrieval
    - **LLM Extraction**: Automated claim identification
    - **Position Tracking**: Monitor competitor strategies
    - **Evolution Analysis**: Detect message shifts over time
    
    **Data Sources:**
    - Competitor press releases
    - Product websites
    - Medical conference presentations
    - Published clinical data
    
    **Recommended Actions:**
    1. Review strategy shift alerts weekly
    2. Analyze strong efficacy claims monthly
    3. Export positioning data for brand planning
    4. Monitor emerging themes in competitor messaging
    """)
    
    # System stats
    st.markdown("---")
    st.markdown("**📊 System Statistics**")
    
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric("Total Documents", f"{total_docs:,}")
        st.metric("Competitors Tracked", competitors)
    
    with col2:
        st.metric("Total Claims", f"{total_claims:,}")
        avg_claims = total_claims / max(1, total_docs)
        st.metric("Avg Claims/Doc", f"{avg_claims:.1f}")
    
    with col3:
        st.metric("Strategy Shifts", recent_shifts)
        st.metric("Coverage Score", f"{coverage_score}/100")

# ============================================================================
# FOOTER
# ============================================================================
st.markdown("---")
col1, col2 = st.columns([3, 1])

with col1:
    st.success("✅ Competitor intelligence system operational | Data current")

with col2:
    if st.button("🔄 Refresh Data"):
        st.rerun()

---

##  Tutorial Complete!

### What You've Learned

1.  **Semantic search** with `CORTEX SEARCH SERVICE`
2.  **LLM-based claim extraction** with `AI_COMPLETE()`
3.  **Positioning analysis** using AI
4.  **Message evolution tracking** with window functions
5.  **Competitive intelligence** dashboards

### Key SQL Techniques

- **`CORTEX SEARCH`** for semantic document retrieval
- **`LATERAL FLATTEN()`** to parse LLM-generated lists
- **`LAG()` and `LEAD()`** for trend analysis
- **`LISTAGG()`** to aggregate text across documents
- **`ARRAY_CONSTRUCT()`** for structured data

### Next Steps for Production

1. **Connect real data sources** (competitor websites, press releases, SEC filings)
2. **Automate document ingestion** (web scraping, RSS feeds, APIs)
3. **Expand search service** to include more document types
4. **Add automated alerts** for new competitor claims
5. **Integrate with brand strategy** planning tools
6. **Create competitive gap reports** (what they claim vs what we claim)

---

**Congratulations!** You've built a production-ready competitive intelligence system that can automatically analyze competitor messaging, extract claims, and identify strategic positioning shifts.

**Estimated runtime**: (varies by data size)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `COMPETITOR_INTEL_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS COMPETITOR_INTEL_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
